In [1]:
import json
import os
import random

In [2]:
with open("template_question.md", 'r', encoding='utf-8') as file:
    lines = file.readlines()
wild_cards: dict = json.loads("\n".join(lines[:35]))
wild_cards = {"{" + num + "}": value for num, value in wild_cards.items()}
wild_marks = ["{" + str(num) + "}" for num in range(1, 12)]
template_questions: list[str] = []
for line in lines[35:]:
    if not line.startswith("#") and line.strip() != "":
        template_questions.append(line)

In [3]:
base_questions: list[str] = []
def wild_card_replace(question: str) -> list[str]:
    result: list[str] = []
    for mark in wild_cards:
        if mark in question:
            cards = wild_cards[mark]
            for card in cards:
                result.extend(wild_card_replace(question.replace(mark, card)))
    if len(result) == 0:
        result.append(question)
    return result
for template_question in template_questions:
    base_question = str(template_question)
    base_questions.extend(wild_card_replace(base_question))
with open("base_question.md", 'w', encoding='utf-8') as file:
    file.writelines(base_questions)

In [4]:
with open("uni_data.json", 'r', encoding='utf-8') as file:
    uni_data = json.loads(file.read())
target_school = {
    "UET": "Đại học công nghệ - Đại học quốc gia Hà Nội", 
    "PTIT": "Học viện Công nghệ Bưu chính Viễn thông",
    "HUST": "Đại học Bách khoa Hà Nội",
    "FTU": "Đại học Ngoại thương",
    "NEU": "Đại học Kinh tế Quốc dân"
}
school_department_mapping: dict[str, list[str]] = {}
for school_acronym in target_school.keys():
    for school_data in uni_data:
        if school_data["acronym"] == school_acronym:
            departments = [item["name"] for item in school_data["universityMajors"]]
            school_department_mapping[school_acronym] = departments
valid_schools = list(school_department_mapping.keys())

generated_questions: list[str] = []
N_COMBINE = 5
N_DEPARTMENT = 2
N_SCHOOL = 2
N_SCORE = 2
N_SUBJECT = 2
SUBJECTS = [
    "Giải tích 1",
    "Vật lý đại cương",
    "Đại số tuyến tính",
    "Triết học Mác-Lênin",
    "Tư tưởng Hoog Chí Minh"
]
def generate_single(question: str) -> list[str]:
    result: list[str] = []
    if "{school}" in question and "{department}" in question:
        for _ in range(N_COMBINE):
            school = random.choice(valid_schools)
            department = random.choice(school_department_mapping[school])
            if random.random() > 0.333:
                school = target_school[school]
            varied_question = question.replace("{school}", school).replace("{department}", department)
            result.extend(generate_single(varied_question))
    elif "{school}" in question and "{school_sub}" in question:
        for _ in range(N_SCHOOL):
            school = random.choice(valid_schools)
            sub_school = school
            while sub_school == school:
                sub_school = random.choice(valid_schools)
            if random.random() > 0.333:
                school = target_school[school]
                sub_school = target_school[sub_school]
            varied_question = question.replace("{school}", school).replace("{school_sub}", sub_school)
            result.extend(generate_single(varied_question))
    elif "{school}" in question:
        for _ in range(N_SCHOOL):
            school = random.choice(valid_schools)
            if random.random() > 0.333:
                school = target_school[school]
            varied_question = question.replace("{school}", school)
            result.extend(generate_single(varied_question))
    elif "{department}" in question:
        for _ in range(N_COMBINE):
            school = random.choice(valid_schools)
            department = random.choice(school_department_mapping[school])
            varied_question = question.replace("{department}", department)
            result.extend(generate_single(varied_question))
    elif "{subject}" in question:
        for _ in range(N_SUBJECT):
            subject = random.choice(SUBJECTS)
            varied_question = question.replace("{subject}", subject)
            result.extend(generate_single(varied_question))
    elif "{score_1}" in question:
        for _ in range(N_SCORE):
            score = random.choice(range(20, 31))
            if score <= 29:
                score = str(score) + random.choice([".5", ".3", ".8", ".25", ".75"])
            varied_question = question.replace("{score_1}", str(score))
            result.extend(generate_single(varied_question))
    elif "{score_2}" in question:
        for _ in range(N_SCORE):
            score = random.choice(range(50, 100))
            if score <= 100:
                score = str(score) + random.choice([".5", ".3", ".8", ".25", ".75"])
            varied_question = question.replace("{score_2}", str(score))
            result.extend(generate_single(varied_question))
    if len(result) == 0:
        result.append(question)
    return result
for base_question in base_questions:
    generated_questions.extend(generate_single(base_question))
with open("generated_question.md", 'w', encoding='utf-8') as file:
    file.writelines(generated_questions)

In [5]:
with open("question_final.md", 'w', encoding='utf-8') as file:
    shuffled_list = list(generated_questions)
    random.shuffle(shuffled_list)
    file.writelines(shuffled_list)